<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Assignment_3_Continuous_Delivery_for_Machine_Learning_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install fastapi uvicorn onnxruntime numpy

In [7]:
%%writefile app.py
from fastapi import FastAPI
import onnxruntime as ort
import numpy as np

app = FastAPI()

# Load ONNX model
session = ort.InferenceSession("model.onnx")

@app.get("/")
def home():
    return {"message": "ML Sentiment API Running"}

@app.post("/predict")
def predict():
    import numpy as np

    # Dummy input (4 features for iris)
    input_data = np.array([[5.1, 3.5, 1.4, 0.2]], dtype=np.float32)

    result = session.run(None, {"input": input_data})

    return {"prediction": result[0].tolist()}

Overwriting app.py


In [8]:
!pip install scikit-learn skl2onnx

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Load sample data
X, y = load_iris(return_X_y=True)

# Train simple model
model = LogisticRegression()
model.fit(X, y)

# Convert to ONNX
initial_type = [('input', FloatTensorType([None, 4]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)

# Save model
with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("ONNX model created successfully!")

ONNX model created successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [31]:
import subprocess

process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
)

In [32]:
import requests

response = requests.post("http://127.0.0.1:8000/predict")
print(response.json())

{'prediction': [0]}


In [28]:
!fuser -k 8000/tcp

8000/tcp:             2968


In [29]:
!uvicorn app:app --host 0.0.0.0 --port 8000

INFO:     Started server process [3735]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3735]


1. Importance of CI/CD in ML
CI/CD helps automate building, testing, and deploying ML models.
•	CI (Continuous Integration): checks code and model updates automatically
•	CD (Continuous Delivery): deploys models quickly and safely
- Helps ML projects by:
•	Detecting errors early
•	Managing model updates
•	Handling data changes

2. Packaging ML model into container
Process:
1.	Build ML model
2.	Create API (Flask/FastAPI)
3.	Write Dockerfile
4.	Add dependencies
5.	Build and run container
- Benefits:
•	Runs same everywhere
•	Easy deployment
•	Scalable

3. Blue-Green vs Canary Deployment
•	Blue-Green:
o	Two environments (old + new)
o	Switch traffic instantly
- Safe rollback
- Needs more resources
•	Canary:
o	Release to small users first
- Gradual testing
- Slower rollout
- Best for healthcare:
Blue-Green → safer and quick rollback

4. Important CI/CD Tests
1.	Unit tests → check code works
2.	Model accuracy test → ensure performance
3.	Input validation test → handle wrong inputs
4.	Integration test → API + model works together

5. Monitoring & Logging
•	Monitoring tracks performance (accuracy, errors)
•	Logging records system activity
- Helps:
•	Detect issues
•	Improve model
•	Maintain reliability